# Week 2, day 4 (afternoon) — Worksheet 05 SOLUTIONS: built-in functions   (L04)

Every cell below was executed on the same Python the lab ships; the quoted
output is real, including the error message in Q3 and the empty second pass
in Q10.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 05 — Built-in functions. Run this once.
words = ["banana", "fig", "apple", "kiwi"]
sales = [120, 340, 90, 500, 210]
monthly = [120, 340, 90, 500, 210]   # a spare copy, sorted in place in Q4
regions = ["East", "West", "Central", "North"]
raw = ["12", "7", "23"]

print("words:  ", words)
print("sales:  ", sales)
print("regions:", regions)
print("raw:    ", raw)

PART A — Measuring, inspecting, converting

### Question 1

Warm-up — len and type. -> `4`, `6`, `2`; then `<class 'list'>`, `<class 'tuple'>`, `<class 'set'>`, `<class 'dict'>`.

`len()` works on anything with a size and means the same thing each time:
number of items. For a string that is characters; for a dictionary it is
PAIRS, not keys plus values.

`type()` is how you settle an argument about what you are actually
holding. Reach for it the moment a structure behaves in a way you did not
expect — it is usually faster than re-reading the code that built it.

In [ ]:
print(len(words))
print(len("banana"))       # a string has a length too
print(len({"a": 1, "b": 2}))   # a dict's length is its number of pairs

print(type(words))
print(type((1, 2)))
print(type({1, 2}))
print(type({"a": 1}))

### Question 2

max, min, sum, and an average. -> `500`, `90`, `1260`, `252.0`, `kiwi`.

There is no built-in average; `sum(x) / len(x)` is it. The result is
`252.0` rather than `252` because `/` always produces a float.

`max(words)` returned `kiwi`, not the longest word. On strings these
functions compare alphabetically, so `max` means "last in dictionary
order". If you wanted the longest, that is `max(words, key=len)` — the
same `key=` idea as Q5.

In [ ]:
print(max(sales))
print(min(sales))
print(sum(sales))
print(sum(sales) / len(sales))   # there is no built-in average

print(max(words))
# max() on strings compares them alphabetically, not by length --
# "kiwi" wins because k comes last, even though "banana" is longer.

### Question 3

Type conversion. -> `19`, `7.0`, `99 bottles`, then `ValueError: invalid literal for int() with base 10: 'twelve'`.

The first line is the one that matters. `raw[0] + raw[1]` on the untouched
strings would have produced `'127'` — silently concatenating instead of
adding, with no error to warn you. Converting first is the difference
between `19` and nonsense.

`int()` parses strings that look like integers. `'twelve'` does not, and
it raises rather than guessing — the same refusal to guess you saw from
`KeyError` in worksheet 04. Anything read from a file or a form is text
until you convert it, and that conversion is where bad input announces
itself.

In [ ]:
print(int(raw[0]) + int(raw[1]))   # "12" + "7" would have been "127"
print(float("3.5") * 2)
print(str(99) + " bottles")        # 99 + " bottles" would have been an error

# This is SUPPOSED to raise. int() converts strings that LOOK like integers.
print(int("twelve"))

PART B — Ordering things

### Question 4

sorted() vs .sort(). -> `[90, 120, 210, 340, 500]`, then `[120, 340, 90, 500, 210]` (untouched), then `[90, 120, 210, 340, 500]`.

Two lines that look alike and have opposite consequences. `sorted()`
returned a new list and left `monthly` alone; `.sort()` returned `None`
and rearranged `monthly` permanently.

Default to `sorted()`. It is the one that cannot lose anything and the one
whose result you can assign — which is worksheet 01 Q9's trap seen from
the other side.

In [ ]:
print(sorted(monthly))   # a NEW sorted list
print(monthly)           # ...and the original is untouched

monthly.sort()           # rearranges in place, returns None
print(monthly)

# Use sorted() when you need the original order kept -- which is most of the
# time. Use .sort() only when you genuinely want to replace it.
# Note `sales` is deliberately left alone here; Q7 and Q9 rely on its order.

### Question 5

sorted with `reverse=` and `key=`. -> `['apple', 'banana', 'fig', 'kiwi']`, `['kiwi', 'fig', 'banana', 'apple']`, `['fig', 'kiwi', 'apple', 'banana']`, and `words` is still `['banana', 'fig', 'apple', 'kiwi']`.

`key=len` sorts by the length of each word rather than its content, so
`fig` (3) leads and `banana` (6) trails. The function you pass to `key` is
applied to every item and its RESULT is what gets compared — the items
themselves come back unchanged, which is why you get words and not
numbers.

The final print is the point of the question: three sorts, and the
original list is exactly as it started.

In [ ]:
print(sorted(words))                 # alphabetical
print(sorted(words, reverse=True))   # alphabetical, backwards
print(sorted(words, key=len))        # by length instead of by content

print(words)   # sorted() never touched it

PART C — Lazy iterators, and the trap they all share

### Question 6

range is a recipe, not a list. -> `range(0, 5)`, `[0, 1, 2, 3, 4]`, `[2, 4, 6, 8]`, `[3, 2, 1]`.

The first print shows `range(0, 5)` rather than any numbers because
`range` stores start, stop and step and produces values only when asked.
`range(1000000)` therefore costs nothing until you iterate it.

Stop is excluded, exactly as in a slice — `range(2, 10, 2)` stops at 8.
A negative step counts down, and needs the start above the stop or you get
nothing at all, silently.

In [ ]:
print(range(5))          # prints range(0, 5) -- a recipe, not the numbers
print(list(range(5)))    # list() makes it produce them

print(list(range(2, 10, 2)))   # start, stop, step -- stop is excluded
print(list(range(3, 0, -1)))   # a negative step counts down

# range() does not build a list of numbers. It stores start/stop/step and
# hands out values only when something asks. list() is that something.

### Question 7

reversed and zip. -> a `<list_reverseiterator object at 0x...>` (the address differs every run), then `['kiwi', 'apple', 'fig', 'banana']`, then 4 pairs, then `4 4 5`.

LOOK AT `4 4 5`. There are five sales figures and four regions, and `zip`
produced four pairs — the `210` was dropped. No error, no warning: zip
stops the moment the shortest input runs out.

That is the most dangerous line on this sheet. Mismatched lengths usually
mean something upstream is wrong, and zip's response is to quietly discard
the evidence. Compare lengths before you zip anything you did not build
yourself.

In [ ]:
print(reversed(words))          # a lazy object again
print(list(reversed(words)))

pairs = list(zip(regions, sales))
print(pairs)
print(len(pairs), len(regions), len(sales))
# zip stops as soon as the SHORTEST input runs out, so the fifth sales
# figure is silently dropped. No error, no warning -- just missing data.

### Question 8

enumerate. -> `1. banana`, `2. fig`, `3. apple`, `4. kiwi`.

`enumerate` hands you `(index, item)` on each pass, so you get a counter
without maintaining one — no `i = 0` before the loop and no `i += 1`
inside it, which is where off-by-one bugs come from.

`start=1` is for display, where humans expect the first row to be 1. It
changes only the number reported, never which item you get.

In [ ]:
for position, word in enumerate(words, start=1):
    print(f"{position}. {word}")

### Question 9

The one-shot trap. -> the four pairs, then `[]`.

Nothing consumed the data in between and nothing failed. The first
`list(report)` walked the zip to its end, and a zip does not rewind — so
the second call found nothing left and correctly returned an empty list.

`zip`, `map`, `filter`, `reversed` and `enumerate` all behave this way.
`range` does NOT: it can be iterated as often as you like. The fix is to
materialise once — `pairs = list(zip(...))` — and reuse `pairs`.

This one is nasty because the symptom, an empty result, looks like a data
problem rather than a consumed-iterator problem, and it sends people
hunting in entirely the wrong place.

In [ ]:
report = zip(regions, sales)

print(list(report))   # everything is there
print(list(report))   # ...and now it is empty

# zip, map, filter, reversed and enumerate are ONE-SHOT. Consuming one
# exhausts it, and it does not rewind. If you need the data twice, store the
# list once (`pairs = list(zip(...))`) and reuse that.

### Question 10

Stretch — map/filter vs comprehensions. -> `[132.0, 374.00000000000006, 99.00000000000001, 550.0, 231.00000000000003]` and `[340, 500, 210]`, then exactly the same two lists again.

Both spellings give identical results, which is the point: nothing here is
a matter of capability, only of readability. Most Python code uses the
comprehension, because the condition reads left to right without a
`lambda` standing between you and it — but `map`/`filter` are common
enough in existing code that you need to recognise them on sight.

AND THE UNTIDY NUMBERS ARE REAL. `374.00000000000006` is binary floating
point, not a bug in your code: `340 * 1.1` has no exact representation, for
the same reason 1/3 has no exact decimal. Never compare two floats with
`==`, and round only at the point where you display them.

In [ ]:
print(list(map(lambda n: n * 1.1, sales)))
print(list(filter(lambda n: n > 200, sales)))

# the same two results, written as comprehensions
print([n * 1.1 for n in sales])
print([n for n in sales if n > 200])

# All four lines are correct. Most Python people reach for the
# comprehension, because the condition reads left to right without a
# lambda in the way -- but map/filter are worth recognising in code you
# did not write.